In [ ]:
!pip install -U transformers datasets accelerate evaluate rouge_score

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import re
import html
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset, Dataset
from evaluate import load as load_metric

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df_training = pd.read_csv("/content/drive/MyDrive/AI_lecture_summarizer/lecture_dataset_train_v3.csv")
df_testing = pd.read_csv("/content/drive/MyDrive/AI_lecture_summarizer/lecture_dataset_test_v3.csv")
display(df_training.head())
display(df_testing.head())

,lecture_id,title,chunk_id,input,label
0,2bi4ha2ejr4or42riqxqpyyp4nyr6q2l,Building and using ontologies,1,So I want someone to tell me at some point. We...,"OK, I probably already wasted 5 minutes of my ..."
1,2bi4ha2ejr4or42riqxqpyyp4nyr6q2l,Building and using ontologies,2,going to give a classical ontology. Engineerin...,"You know, you know about this, yeah, so an ont..."
2,2bi4ha2ejr4or42riqxqpyyp4nyr6q2l,Building and using ontologies,3,"look very similar on the first site, so you mi...","Anyway, that's the classical ontology engineer..."
3,2bi4ha2ejr4or42riqxqpyyp4nyr6q2l,Building and using ontologies,4,"Game in the US. Ann is now applied, so this is...",You can do beautiful things with rich knowledg...
4,2bi4ha2ejr4or42riqxqpyyp4nyr6q2l,Building and using ontologies,5,the relationship? And so on and so forth. And ...,I'm going to tell you where you're likely to f...


,lecture_id,title,chunk_id,input,label
0,4zb643hqgwxsyy2lo4kmpgzbpuce3ixa,New trends in public service and Citizens Comm...,1,Thank you very much and thank you very much fo...,I would like to talk about new trends in publi...
1,4zb643hqgwxsyy2lo4kmpgzbpuce3ixa,New trends in public service and Citizens Comm...,2,sector and 60% has says that they have sent in...,And the elder people don't have high capabilit...
2,4zb643hqgwxsyy2lo4kmpgzbpuce3ixa,New trends in public service and Citizens Comm...,3,Internet and they are happy to do it and they ...,They will not do it when we provide things fro...
3,4zb643hqgwxsyy2lo4kmpgzbpuce3ixa,New trends in public service and Citizens Comm...,4,it is the main entrance for these social servi...,So and the goal is that 80% of the of the citi...
4,4zb643hqgwxsyy2lo4kmpgzbpuce3ixa,New trends in public service and Citizens Comm...,5,doesn't contain all the data it displayed the ...,They have the data but we display it on the po...


In [ ]:
df_training = df_training.drop(columns = ['lecture_id','title','chunk_id'])
df_testing = df_testing.drop(columns = ['lecture_id','title','chunk_id'])
df_training['label'] = df_training['label'].replace('0', "")
df_testing['label'] = df_testing['label'].replace('0', "")
display(df_training.head())
display(df_testing.head())

,input,label
0,So I want someone to tell me at some point. We...,"OK, I probably already wasted 5 minutes of my ..."
1,going to give a classical ontology. Engineerin...,"You know, you know about this, yeah, so an ont..."
2,"look very similar on the first site, so you mi...","Anyway, that's the classical ontology engineer..."
3,"Game in the US. Ann is now applied, so this is...",You can do beautiful things with rich knowledg...
4,the relationship? And so on and so forth. And ...,I'm going to tell you where you're likely to f...


,input,label
0,Thank you very much and thank you very much fo...,I would like to talk about new trends in publi...
1,sector and 60% has says that they have sent in...,And the elder people don't have high capabilit...
2,Internet and they are happy to do it and they ...,They will not do it when we provide things fro...
3,it is the main entrance for these social servi...,So and the goal is that 80% of the of the citi...
4,doesn't contain all the data it displayed the ...,They have the data but we display it on the po...


In [ ]:
print(df_training['input'][0])

So I want someone to tell me at some point. Well now it's 11. Can someone like give me a not so subtle sign in half an hour? OK, um, so that you can all go to lunch and and everything works as planned. But before I start with the tutorial. There is this box here. Witcher Aiden created and decorated in a typically Irish fashion. Well the purpose of this box is to collect anonymous questions from tutors and and students and well, depending on how successful we are. Maybe also other people who are sitting around questions about the topics that we have addressed at the school and the idea would be that we collect these topics. Today and throughout tomorrow during the keynote and after the keynote tomorrow we have something like a very informal panel, so all the tutors will gather here in the front and we will pick questions and we will spend something like an hour. Trying to answer them. And some of the answers might be less factual than others. So for some of these questions, we might hav

In [ ]:
print(df_training['label'][0])

OK, I probably already wasted 5 minutes of my precious time building and using ontologies. You know, you know about this, yeah, so an ontology is talking about a domain you don't talk about.


In [ ]:
filler_phrases = [
    'thank you', 'thanks everyone', 'bye', 'see you', 'hi everyone', 'hello everyone',
    'welcome to', 'let’s begin', 'let us begin', 'let’s start', 'so yeah', 'you know',
    'um', 'uh', 'ok', 'okay', 'alright', 'right', 'so now', 'so next', 'moving on',
    'as i said', 'as you can see', 'you can see', 'in this slide', 'on this slide',
    'slide shows', 'we will talk about', 'we’re going to talk about', 'i’m going to talk about',
    'let’s talk about', 'talk a little bit', 'for example', 'for instance',
    'basically', 'actually', 'so basically', 'so actually', 'kind of', 'sort of',
    'like i said', 'that’s it', 'that’s all', 'make sense', 'you know what i mean'
]

filler_pattern = re.compile(r'\b(' + '|'.join(map(re.escape, filler_phrases)) + r')\b', re.IGNORECASE)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)

    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\([^)]*\)", "", text)

    text = filler_pattern.sub("", text)

    text = re.sub(r"\.{2,}", ".", text)
    text = re.sub(r"\s{2,}", " ", text)

    text = text.strip()
    return text

df_training['clean_input'] = df_training['input'].apply(clean_text)
df_training['clean_label'] = df_training['label'].apply(clean_text)
df_testing['clean_input'] = df_testing['input'].apply(clean_text)
df_testing['clean_label'] = df_testing['label'].apply(clean_text)
df_training.head()

,input,label,clean_input,clean_label
0,So I want someone to tell me at some point. We...,"OK, I probably already wasted 5 minutes of my ...",So I want someone to tell me at some point. We...,", I probably already wasted 5 minutes of my pr..."
1,going to give a classical ontology. Engineerin...,"You know, you know about this, yeah, so an ont...",going to give a classical ontology. Engineerin...,", about this, yeah, so an ontology is talking ..."
2,"look very similar on the first site, so you mi...","Anyway, that's the classical ontology engineer...","look very similar on the first site, so you mi...","Anyway, that's the classical ontology engineer..."
3,"Game in the US. Ann is now applied, so this is...",You can do beautiful things with rich knowledg...,"Game in the US. Ann is now applied, so this is...",You can do beautiful things with rich knowledg...
4,the relationship? And so on and so forth. And ...,I'm going to tell you where you're likely to f...,the relationship? And so on and so forth. And ...,I'm going to tell you where you're likely to f...


In [ ]:
model_name = 'facebook/bart-large-cnn'
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = Dataset.from_pandas(df_training[["clean_input", "clean_label"]])
test_dataset  = Dataset.from_pandas(df_testing[["clean_input", "clean_label"]])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
MAX_INPUT_LENGTH = 1024
MAX_LABEL_LENGTH = 256

def tokenize_function(batch):
    model_inputs = tokenizer(
        batch["clean_input"],
        max_length=MAX_INPUT_LENGTH,
        padding="max_length",
        truncation=True,
    )

    labels = tokenizer(
        batch["clean_label"],
        max_length=MAX_LABEL_LENGTH,
        padding="max_length",
        truncation=True,
    )

    labels["input_ids"] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized  = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/62679 [00:00<?, ? examples/s]

Map:   0%|          | 0/8196 [00:00<?, ? examples/s]

In [ ]:
train_tokenized.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)
test_tokenized.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
example = train_tokenized[0]
print("Input tokens:", len(example["input_ids"]))
print("Label tokens:", len(example["labels"]))
print("First 20 input IDs:", example["input_ids"][:20])
print("First 20 label IDs:", example["labels"][:20])

Input tokens: 1024
Label tokens: 256
First 20 input IDs: tensor([   0, 2847,   38,  236,  951,    7, 1137,  162,   23,  103,  477,    4,
        2647,  122,   24,   18,  365,    4, 2615,  951])
First 20 label IDs: tensor([    0,     6,    38,  1153,   416, 14260,   195,   728,     9,   127,
         9761,    86,   745,     8,   634, 25099, 20643,     4,  2156,    59])


In [ ]:
output_dir = "/content/drive/MyDrive/AI_lecture_summarizer/Bart_checkpoints_large"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    overwrite_output_dir=False,
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="rougeLsum",
    greater_is_better=True,
    eval_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    predict_with_generate=True,
    generation_max_length=256,
    generation_num_beams=4,
    learning_rate=4e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",
    adam_epsilon=1e-8,
    max_grad_norm=1.0,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    fp16=True,
    resume_from_checkpoint=True
)

In [ ]:
rouge = load_metric("rouge")

def compute_metrics(eval_pred):
    try:
        preds, labels = eval_pred
        preds = np.array(preds)
        labels = np.array(labels)
        preds = np.clip(preds, 0, tokenizer.vocab_size - 1)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
        final_result = {}
        for key, value in result.items():
            if hasattr(value, "mid"):
                final_result[key] = value.mid.fmeasure * 100
            else:
                final_result[key] = float(value) * 100
        final_result["gen_len"] = np.mean([len(pred.split()) for pred in decoded_preds])
        return {k: round(v, 4) for k, v in final_result.items()}
    except Exception as e:
        print(f"[WARNING] Metric computation failed: {e}")
        return {}



In [ ]:
small_eval_dataset = test_tokenized.select(range(1000))

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=small_eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-1674030562.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
try:
    trainer.train(resume_from_checkpoint=True)
except Exception as e:
    print(f"[WARNING] Training interrupted: {e}")
    trainer.save_model()
    trainer.save_state()

You are resuming training from a checkpoint trained with 4.57.1 of Transformers but your current version is 4.57.2. This is not recommended and could yield to errors or unwanted behaviors.
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
4,0.125100,0.559163,35.888200,24.633500,28.826000,28.864800,112.405000


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
